In [ ]:
import sys
sys.path.append("..")  # Adds the parent directory (src) to the Python path

In [ ]:
from backend.backend.utils import pod_parser

In [ ]:
import requests
SERVER_URL = "http://127.0.0.1:8008"
#SERVER_URL = "http://192.168.2.239"
# gather the podcast slug, episode guid, and audio file link for transcription
podcasts = requests.get(f"{SERVER_URL}/api/podcasts/")
podcasts = podcasts.json()
for i, p in enumerate(podcasts):
    print(f"{i}: {p['title']} ({p['slug']})")

In [ ]:
def get_new_episodes(podcast):
    """Get the new episodes of a podcast from the RSS feed.

    Parameters
    ----------
    podcast : dict
        The podcast dictionary.

    Returns
    -------
    list
        A list of new episodes.
    """
    episodes_all = pod_parser.parse_channel(podcast["rss"])["audioitem_set"]
    episodes_db = [entry["guid"] for entry in podcast["audioitem_set"]]
    new_eps = []
    for episode in episodes_all:
        if episode["guid"] not in episodes_db:
            new_eps.append(episode)
        else:
            break
    return new_eps



In [ ]:
def get_episode_by_guid(podcast_slug, podcasts, episode_guid):
    """return an episode of the podcast with the given guid by querying the server and reading the RSS feed
    
    Parameters
    ----------
    podcast_slug : str
        The slug of the podcast.
    podcasts : list
        List of podcast objects.
    episode_guid : str
        The guid of the episode to add.

    Returns
    -------
    dict
        The episode dictionary.
    """
    podcast = [pod for pod in podcasts if pod["slug"] == podcast_slug][0]
    episodes_all = pod_parser.parse_channel(podcast["rss"])
    for episode in episodes_all["audioitem_set"]:
        if episode["guid"] == episode_guid:
            return episode
    return None


In [ ]:
# get the uuid field of each segmentation object in each segmentation_set for each transcription in transcription_set and each audioitem in audioitem_set and each podcast in podcasts
segmentation_uuids = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == 'spaCy':
                    segmentation_uuids.append(segmentation['uuid'])
len(segmentation_uuids)

In [ ]:
def get_episodes_wo_transcription(podcasts):
    """Get the episodes already in the database without transcription for each podcast.

    Parameters
    ----------
    podcasts : list
        A list of podcasts.

    Returns
    -------
    list
        A list of episodes without transcription for each podcast.
    """
    episodes_wo_transcription = []
    for podcast in podcasts:
        for audioitem in podcast['audioitem_set']:
            # check if audioitem['transcription_set'] is empty
            if len(audioitem['transcription_set']) == 0:
                episodes_wo_transcription.append((audioitem, podcast))
    return episodes_wo_transcription


In [ ]:
import os

def download_audio_file(new_ep, podcast_slug):
    current_dir = os.path.dirname(os.getcwd()) # get parent of current directory
    media_dir = current_dir + "/media"

    link = new_ep.get("audio_link")
    guid = new_ep.get("guid")
    fileroot = f"{media_dir}/{podcast_slug}_{guid}"
    # check for wav file
    if not os.path.exists(f"{fileroot}.wav"):
        # check for mp3 file
        if not os.path.exists(f"{fileroot}.mp3"):
            # download mp3 file
            print(f"Downloading {fileroot}")
            !curl -o '{fileroot + ".mp3"}' -L -J '{link}'
        # convert mp3 to wav
        !ffmpeg -i '{fileroot + ".mp3"}' -vn -acodec pcm_s16le -ar 16000 -ac 1 '{fileroot + ".wav"}'
        print(f"Downloaded {fileroot}")
    else:
        print(f"File {fileroot} already exists")


    filepath = fileroot + ".wav"
    return filepath

In [ ]:
def add_visibility_to_utterances(utterances, visibility):
    """Add the visibility field to each utterance.

    Parameters
    ----------
    utterances : list
        A list of utterances.
    visibility : JSON, 1 for all, or a list of the visible qualifiers
        The visibility of the utterances.

    Returns
    -------
    list
        A list of utterances with the visibility field.
    """
    for utterance in utterances:
        utterance["visibility"] = visibility
    return utterances

Fetch new episodes for existing podcasts

In [ ]:
from transcribe_file import get_transcription
import sentence_splitter
import spacy

# update podcasts that are already in the database with new episodes
for podcast in podcasts:
    slug = podcast["slug"]
    print(slug)
    
    new_eps = get_new_episodes(podcast)
    files = []
    for ep in new_eps:
        print(ep.get("title"))
        # get file
        file = download_audio_file(ep, slug)
        # run transcription
        lang = podcast["language"][0:2]
        script = get_transcription(file, language=lang if lang != "nb" else "no", model_size="large")

        # post episode to api
        res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/episodes/", json=ep)
        print(res.status_code)

        # post transcription to api
        transcription_dict = script["transcription"]
        guid = "_".join(file.split("/")[-1].split("_")[1:]).split(".")[0]
        transcription_dict["guid"] = guid
        res = requests.post(f"{SERVER_URL}/api/transcriptions/", json=transcription_dict)
        print(res.status_code)

        # post whisper default segmentation
        segmentation_dict = script["segmentation"]
        segmentation_dict["utterance_set"] = add_visibility_to_utterances(segmentation_dict["utterance_set"], 1)
        trans_uuid = res.json().get("uuid")
        segmentation_dict["uuid"] = trans_uuid
        res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict)
        print(res.status_code)

        # use spacy to split the text into sentences
        try :
            utterances = sentence_splitter.sentence_splitter(transcription_dict, "en_core_web_lg" if lang == "en" else "nb_core_news_lg")

            segmentation_dict_spacy = {
                "uuid": trans_uuid,
                "name": "spaCy",
                "segmentor": {"name": "spaCy", "version": spacy.__version__},
                "utterance_set": add_visibility_to_utterances([utterance for utterance in utterances if utterance.get("text") != ""], 1)
            }

            res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict_spacy)
            print("spaCy", res.status_code)
        except:
            print("spaCy failed")



### Add new episodes to podcast by their GUID always found in their RSS (transcription runs separately)

In [ ]:
#podcast_slug = "leger-om-livet"
#episode_guids = [
#    "63c5c8969ae24b0011e91c4f", #92
#    "62b0e8404f1d1f001403f20c", #73
#    "6005f549bb8e664143201208", #10
#    "601870631dd62059644186ef", #12
#    "6058dba64277fb690eef8d6d", #19
#    "606b3c33b9b65c05c209bab0", #20
#    "601ea131d782921f4ea19d0c", #13
#    "602aa8d050f08c637903c206", #14
#]

for guid in episode_guids:
    ep = get_episode_guid(podcast_slug, podcasts, guid)
    # post episode to api
    res = requests.post(f"{SERVER_URL}/api/podcasts/{podcast_slug}/episodes/", json=ep)
    print(res.status_code)

Add a new podcast

In [ ]:
new_eps = ["https://feeds.soundcloud.com/users/soundcloud:users:326345351/sounds.rss"]

In [ ]:
import requests
# Add each podcast in RSS to database via API
# the API will automatically download some number of recent episodes

for podcast in new_eps:
  print(podcast)
  res = requests.post(f"{SERVER_URL}/api/podcasts/", data={
    "rss": podcast
  })
  print(res.status_code)

In [ ]:
eps = get_episodes_wo_transcription(podcasts)
len(eps)

In [ ]:
len(eps)

In [ ]:
from transcribe_file import get_transcription
import sentence_splitter
import spacy
from json import JSONDecodeError

eps = get_episodes_wo_transcription(podcasts)

# update podcasts that are already in the database with new episodes

for episode in eps:
    podcast = episode[1]
    ep = episode[0]
    slug = podcast.get("slug")
    print(ep.get("title"))
    # get file
    file = download_audio_file(ep, slug)
    # run transcription
    lang = podcast["language"][0:2]
    script = get_transcription(file, language=lang if lang != "nb" else "no", model_size="large")

    # post transcription to api
    transcription_dict = script["transcription"]
    guid = "_".join(file.split("/")[-1].split("_")[1:]).split(".")[0]
    transcription_dict["guid"] = guid
    res = requests.post(f"{SERVER_URL}/api/transcriptions/", json=transcription_dict)
    print(res.status_code)

    # post whisper default segmentation
    segmentation_dict = script["segmentation"]
    trans_uuid = res.json().get("uuid")
    segmentation_dict["uuid"] = trans_uuid
    res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict)
    print(res.status_code)

    # use spacy to split the text into sentences
    spacy_dict = {"en": "en_core_web_lg", "no": "nb_core_news_lg", "de": "de_dep_news_trf", "se": "sv_core_news_lg", "da": "da_core_news_trf"}
    try :
        utterances = sentence_splitter.sentence_splitter(transcription_dict, spacy_dict[lang])

        segmentation_dict_spacy = {
            "uuid": trans_uuid,
            "name": "spaCy",
            "segmentor": {"name": "spaCy", "version": spacy.__version__},
            "utterance_set": [utterance for utterance in utterances if utterance.get("text") != ""]
        }

        res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict_spacy)
        print("spaCy", res.status_code)
    except:
        print("spaCy failed")